<a href="https://colab.research.google.com/github/saif-islam-rayhan/Deep-Learning-Project/blob/main/Module11_IMDB_Sentiment_SaifIslamRayhan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Run this first in Colab
!pip install -q datasets transformers sentence-transformers gensim scikit-learn nltk tqdm

import nltk
nltk.download('punkt')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 47.4 MB/s eta 0:00:00


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [2]:
import random
import numpy as np
import pandas as pd
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("GPU available:", torch.cuda.is_available())


GPU available: True


In [3]:
from datasets import load_dataset
from sklearn.model_selection import train_test_split

imdb = load_dataset("imdb")

df_train = pd.DataFrame(imdb['train'])
df_test  = pd.DataFrame(imdb['test'])

train_df, val_df = train_test_split(df_train, test_size=0.1, random_state=SEED, stratify=df_train['label'])
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = df_test.reset_index(drop=True)

print("sizes ->", len(train_df), len(val_df), len(test_df))


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

sizes -> 22500 2500 25000


In [4]:
import re
from nltk.tokenize import word_tokenize

def preprocess(text):
    text = str(text)
    text = re.sub(r'<.*?>', ' ', text)          # HTML tag remove
    text = text.lower()                         # lowercase
    text = re.sub(r'[^a-z0-9\s]', ' ', text)    # punctuation remove (english-focused)
    text = re.sub(r'\s+', ' ', text).strip()    # extra spaces
    return text

# preprocessing
for df in (train_df, val_df, test_df):
    df['text_clean'] = df['text'].apply(preprocess)

train_df[['text', 'text_clean', 'label']].head(3)


,text,text_clean,label
0,"""Algie, the Miner"" is one bad and unfunny sile...",algie the miner is one bad and unfunny silent ...,0
1,This is a complete Hoax...<br /><br />The movi...,this is a complete hoax the movie clearly has ...,0
2,"Nifty little episode played mainly for laughs,...",nifty little episode played mainly for laughs ...,1


# TF–IDF + Logistic Regression

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

# TF-IDF
tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1,2))
X_train_tfidf = tfidf.fit_transform(train_df['text_clean'])
X_val_tfidf   = tfidf.transform(val_df['text_clean'])
X_test_tfidf  = tfidf.transform(test_df['text_clean'])

y_train = train_df['label'].values
y_val   = val_df['label'].values
y_test  = test_df['label'].values

clf_tfidf = LogisticRegression(max_iter=1000, random_state=SEED)
clf_tfidf.fit(X_train_tfidf, y_train)

y_pred_tfidf = clf_tfidf.predict(X_test_tfidf)
acc_tfidf = accuracy_score(y_test, y_pred_tfidf)
prec_tfidf, rec_tfidf, f1_tfidf, _ = precision_recall_fscore_support(y_test, y_pred_tfidf, average='binary')

print("TF-IDF Test -> Acc: %.4f  Prec: %.4f  Rec: %.4f  F1: %.4f" % (acc_tfidf, prec_tfidf, rec_tfidf, f1_tfidf))


TF-IDF Test -> Acc: 0.8935  Prec: 0.8912  Rec: 0.8964  F1: 0.8938


# Word2Vec (gensim) + average embeddings + classifier

In [8]:
from gensim.models import Word2Vec
from nltk.tokenize import word_tokenize
import numpy as np

# tokenize texts (list of token lists)
train_tokens = [word_tokenize(t) for t in train_df['text_clean']]
val_tokens   = [word_tokenize(t) for t in val_df['text_clean']]
test_tokens  = [word_tokenize(t) for t in test_df['text_clean']]

# train Word2Vec on training sentences
w2v_model = Word2Vec(sentences=train_tokens, vector_size=100, window=5, min_count=2, workers=4, seed=SEED)

def sent_vector(tokens, model, dim=100):
    vecs = [model.wv[w] for w in tokens if w in model.wv]
    if len(vecs)==0:
        return np.zeros(dim)
    return np.mean(vecs, axis=0)

X_train_w2v = np.vstack([sent_vector(s, w2v_model) for s in train_tokens])
X_val_w2v   = np.vstack([sent_vector(s, w2v_model) for s in val_tokens])
X_test_w2v  = np.vstack([sent_vector(s, w2v_model) for s in test_tokens])

clf_w2v = LogisticRegression(max_iter=1000, random_state=SEED)
clf_w2v.fit(X_train_w2v, y_train)

y_pred_w2v = clf_w2v.predict(X_test_w2v)
acc_w2v = accuracy_score(y_test, y_pred_w2v)
prec_w2v, rec_w2v, f1_w2v, _ = precision_recall_fscore_support(y_test, y_pred_w2v, average='binary')

print("Word2Vec Test -> Acc: %.4f  Prec: %.4f  Rec: %.4f  F1: %.4f" % (acc_w2v, prec_w2v, rec_w2v, f1_w2v))

Word2Vec Test -> Acc: 0.8308  Prec: 0.8330  Rec: 0.8275  F1: 0.8302


# BERT embeddings (Sentence-Transformers) + classifier

In [10]:
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

sbert_model = SentenceTransformer('all-MiniLM-L6-v2')

def embed_texts(texts, model, batch_size=64):
    embeddings = []
    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i+batch_size]
        emb = model.encode(batch, show_progress_bar=False)
        embeddings.append(emb)
    return np.vstack(embeddings)

X_train_bert = embed_texts(train_df['text_clean'].tolist(), sbert_model)
X_val_bert   = embed_texts(val_df['text_clean'].tolist(), sbert_model)
X_test_bert  = embed_texts(test_df['text_clean'].tolist(), sbert_model)

clf_bert = LogisticRegression(max_iter=1000, random_state=SEED)
clf_bert.fit(X_train_bert, y_train)

y_pred_bert = clf_bert.predict(X_test_bert)
acc_bert = accuracy_score(y_test, y_pred_bert)
prec_bert, rec_bert, f1_bert, _ = precision_recall_fscore_support(y_test, y_pred_bert, average='binary')

print("BERT Test -> Acc: %.4f  Prec: %.4f  Rec: %.4f  F1: %.4f" % (acc_bert, prec_bert, rec_bert, f1_bert))


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

  0%|          | 0/352 [00:00<?, ?it/s]

  0%|          | 0/40 [00:00<?, ?it/s]

  0%|          | 0/391 [00:00<?, ?it/s]

BERT Test -> Acc: 0.8186  Prec: 0.8221  Rec: 0.8132  F1: 0.8176


In [11]:
results = pd.DataFrame([
    ['TF-IDF', acc_tfidf, prec_tfidf, rec_tfidf, f1_tfidf],
    ['Word2Vec', acc_w2v, prec_w2v, rec_w2v, f1_w2v],
    ['BERT (SBERT)', acc_bert, prec_bert, rec_bert, f1_bert]
], columns=['Method','Accuracy','Precision','Recall','F1'])

results


,Method,Accuracy,Precision,Recall,F1
0,TF-IDF,0.89348,0.891195,0.8964,0.893790
1,Word2Vec,0.82924,0.829003,0.8296,0.829301
2,BERT (SBERT),0.81860,0.822078,0.8132,0.817615
